In [1]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)

In [2]:
cohort = pd.read_csv("../data/hf_dataset/sepsis_cohort.csv")
diagnosis = pd.read_csv("../data/hf_dataset/sepsis_diagnoses_icd.csv")
timeseries = pd.read_csv("../data/hf_dataset/sepsis_timeseries_hourly.csv")
time_comparison = pd.read_csv("../data/hf_dataset/sepsis_time_comparison.csv")

datasets = {
    "Cohort": cohort,
    "Diagnosis": diagnosis,
    "Timeseries": timeseries,
    "Time Comparison": time_comparison
}
for name , df in datasets.items():
    print("=" * 80)
    print(name)
    print("\nshape:")
    print(df.shape)
    print("\ncolumns:")
    print(df.columns.tolist())
    print("\ninfo:")
    print(df.info())


Cohort

shape:
(32971, 12)

columns:
['subject_id', 'hadm_id', 'stay_id', 'icu_intime', 'icu_outtime', 'sepsis_time', 'suspected_infection_time', 'window_end_uncapped', 'window_end', 'gender', 'age', 'n_hours']

info:
<class 'pandas.DataFrame'>
RangeIndex: 32971 entries, 0 to 32970
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   subject_id                32971 non-null  int64
 1   hadm_id                   32971 non-null  int64
 2   stay_id                   32971 non-null  int64
 3   icu_intime                32971 non-null  str  
 4   icu_outtime               32971 non-null  str  
 5   sepsis_time               32971 non-null  str  
 6   suspected_infection_time  32971 non-null  str  
 7   window_end_uncapped       32971 non-null  str  
 8   window_end                32971 non-null  str  
 9   gender                    32971 non-null  str  
 10  age                       32971 non-null  i

In [3]:
for name, df in datasets.items():
    print("=" * 80)
    print(name)
    missing = (
        df.isnull().sum().sort_values(ascending=False)
    )
    display(missing.head(20))


Cohort


subject_id                  0
hadm_id                     0
stay_id                     0
icu_intime                  0
icu_outtime                 0
sepsis_time                 0
suspected_infection_time    0
window_end_uncapped         0
window_end                  0
gender                      0
age                         0
n_hours                     0
dtype: int64

Diagnosis


stay_id        0
hadm_id        0
seq_num        0
icd_code       0
icd_version    0
dtype: int64

Timeseries


crp                     850993
bands                   846016
rate_dobutamine         845221
pao2fio2ratio_novent    843070
rate_dopamine           841195
albumin                 837294
rate_epinephrine        831471
bilirubin_total         825268
bilirubin_max           825251
liver                   825251
pao2fio2ratio_vent      799428
inr                     792118
pt                      792103
respiration             790810
ptt                     790605
lactate                 789034
uo_24hr                 784838
calcium                 782512
magnesium               777808
glucose_lab             776253
dtype: int64

Time Comparison


spesis_hour_delta2                            13817
spesis_time_delta2                            13817
diff_hours_delta2                             13817
spesis_hour_sofa2                              7842
spesis_time_sofa2                              7842
diff_hours_sofa2                               7842
sepsis3_endtime                                 734
sepsis3_time_hour_floor                         734
diff_hours_sepsis3                              734
cohort_sepsis_hour_from_intime_floor              0
suspected_infection_time                          0
suspected_infection_hour_from_intime_floor        0
icu_intime                                        0
icu_intime_floor                                  0
stay_id                                           0
cohort_sepsis_time                                0
dtype: int64

In [21]:
feature_inventory = pd.DataFrame({
    "Features":  [],
    "Dataset":  [],
    "Cliniical Category":  [],
    "Prediction Time Available": [],
    "Leak Risk": [],
    "Decision": [],
    "Notes": []
})
feature_inventory


,Features,Dataset,Cliniical Category,Prediction Time Available,Leak Risk,Decision,Notes


In [31]:
feature_inventory = pd.DataFrame({
    "Feature": timeseries.columns,
    "Dataset": "Timeseries",
    "Clinical Category": "",
    "Prediction Time Available": "",
    "Leak Risk": "",
    "Decision": "",
    "Notes": ""
})

feature_inventory.head(15)


,Feature,Dataset,Clinical Category,Prediction Time Available,Leak Risk,Decision,Notes
0,stay_id,Timeseries,,,,,
1,hour,Timeseries,,,,,
2,heart_rate,Timeseries,,,,,
3,resp_rate,Timeseries,,,,,
4,temperature,Timeseries,,,,,
5,sbp,Timeseries,,,,,
6,dbp,Timeseries,,,,,
7,mbp,Timeseries,,,,,
8,spo2,Timeseries,,,,,
9,gcs,Timeseries,,,,,


In [33]:
feature_inventory.rename(columns={"features": "Feature"}, inplace=True)

feature_inventory.head()
feature_inventory.loc[
    feature_inventory["Feature"].isin(
        [
            "heart_rate",
            "resp_rate",
            "temperature",
            "sbp",
            "dbp",
            "mbp",
            "spo2",
            "gcs"
        ]
    ),
    ["Clinical Category", "Decision"]
] = [
    "Vital Sign",
    "Keep"
]

In [34]:
feature_inventory.head(15)

,Feature,Dataset,Clinical Category,Prediction Time Available,Leak Risk,Decision,Notes
0,stay_id,Timeseries,,,,,
1,hour,Timeseries,,,,,
2,heart_rate,Timeseries,Vital Sign,,,Keep,
3,resp_rate,Timeseries,Vital Sign,,,Keep,
4,temperature,Timeseries,Vital Sign,,,Keep,
5,sbp,Timeseries,Vital Sign,,,Keep,
6,dbp,Timeseries,Vital Sign,,,Keep,
7,mbp,Timeseries,Vital Sign,,,Keep,
8,spo2,Timeseries,Vital Sign,,,Keep,
9,gcs,Timeseries,Vital Sign,,,Keep,


In [35]:
vital_signs = [
    "heart_rate",
    "resp_rate",
    "temperature",
    "sbp",
    "dbp",
    "mbp",
    "spo2",
    "gcs"
]

In [36]:
feature_inventory.loc[
    feature_inventory["Feature"].isin(vital_signs),
    "Prediction Time Available"
] = "Yes"
feature_inventory.loc[
    feature_inventory["Feature"].isin(vital_signs),
    "Leak Risk"
] = "Low"
feature_inventory.loc[
    feature_inventory["Feature"].isin(vital_signs),
    "Notes"
] = "Core physiological measurements"

In [37]:
feature_inventory.head(15)

,Feature,Dataset,Clinical Category,Prediction Time Available,Leak Risk,Decision,Notes
0,stay_id,Timeseries,,,,,
1,hour,Timeseries,,,,,
2,heart_rate,Timeseries,Vital Sign,Yes,Low,Keep,Core physiological measurements
3,resp_rate,Timeseries,Vital Sign,Yes,Low,Keep,Core physiological measurements
4,temperature,Timeseries,Vital Sign,Yes,Low,Keep,Core physiological measurements
5,sbp,Timeseries,Vital Sign,Yes,Low,Keep,Core physiological measurements
6,dbp,Timeseries,Vital Sign,Yes,Low,Keep,Core physiological measurements
7,mbp,Timeseries,Vital Sign,Yes,Low,Keep,Core physiological measurements
8,spo2,Timeseries,Vital Sign,Yes,Low,Keep,Core physiological measurements
9,gcs,Timeseries,Vital Sign,Yes,Low,Keep,Core physiological measurements


In [38]:
identifier_features = [
    "stay_id"
]

feature_inventory.loc[
    feature_inventory["Feature"].isin(identifier_features), # .loc helps in updating every feature in identifier feature from feature identifier
    "Clinical Category"
] = "Identifier"

feature_inventory.loc[
    feature_inventory["Feature"].isin(identifier_features),
    "Prediction Time Available"
] = "Yes"

feature_inventory.loc[
    feature_inventory["Feature"].isin(identifier_features),
    "Leak Risk"
] = "None"

feature_inventory.loc[
    feature_inventory["Feature"].isin(identifier_features),
    "Decision"
] = "Remove"

feature_inventory.loc[
    feature_inventory["Feature"].isin(identifier_features),
    "Notes"
] = "Unique patient identifier"

In [42]:
time_features = [ "hour" ]
feature_inventory.loc[ feature_inventory["Feature"].isin(time_features),"Clinical Category"] = "Time"

feature_inventory.loc[ feature_inventory["Feature"].isin(time_features), "Prediction Time Available"] = "yes"
feature_inventory.loc[ feature_inventory["Feature"].isin(time_features), "Leak Risk"] = "Low"
feature_inventory.loc[feature_inventory["Feature"].isin(time_features) , "Decision"] = "keep"
feature_inventory.loc[feature_inventory["Feature"].isin(time_features) ,"Notes"] = "hours since icu admission "
feature_inventory.drop(columns=["Prediction Time available"], inplace=True)
feature_inventory.head(10)

,Feature,Dataset,Clinical Category,Prediction Time Available,Leak Risk,Decision,Notes
0,stay_id,Timeseries,Identifier,Yes,None,Remove,Unique patient identifier
1,hour,Timeseries,Time,yes,Low,keep,hours since icu admission
2,heart_rate,Timeseries,Vital Sign,Yes,Low,Keep,Core physiological measurements
3,resp_rate,Timeseries,Vital Sign,Yes,Low,Keep,Core physiological measurements
4,temperature,Timeseries,Vital Sign,Yes,Low,Keep,Core physiological measurements
5,sbp,Timeseries,Vital Sign,Yes,Low,Keep,Core physiological measurements
6,dbp,Timeseries,Vital Sign,Yes,Low,Keep,Core physiological measurements
7,mbp,Timeseries,Vital Sign,Yes,Low,Keep,Core physiological measurements
8,spo2,Timeseries,Vital Sign,Yes,Low,Keep,Core physiological measurements
9,gcs,Timeseries,Vital Sign,Yes,Low,Keep,Core physiological measurements


In [43]:
print(feature_inventory.columns.tolist())
for col in timeseries.columns:
    print(col)

['Feature', 'Dataset', 'Clinical Category', 'Prediction Time Available', 'Leak Risk', 'Decision', 'Notes']
stay_id
hour
heart_rate
resp_rate
temperature
sbp
dbp
mbp
spo2
gcs
urineoutput_last
urineoutput_sum
wbc
hemoglobin
hematocrit
platelet
creatinine
bun
sodium
potassium
chloride
bicarbonate
calcium
aniongap
glucose_lab
albumin
bilirubin_total
inr
pt
ptt
lactate
bands
crp
magnesium
starttime
endtime
pao2fio2ratio_novent
pao2fio2ratio_vent
rate_epinephrine
rate_norepinephrine
rate_dopamine
rate_dobutamine
meanbp_min
gcs_min
urineoutput_24hr
uo_tm_24hr
uo_24hr
bilirubin_max
creatinine_max
platelet_min
respiration
coagulation
liver
cardiovascular
cns
renal
sofa_score
respiration_24hours
coagulation_24hours
liver_24hours
cardiovascular_24hours
cns_24hours
renal_24hours
sofa_24hours
